# Rhino OBJ → TopologicPy CellComplex & Graph

Before running this notebook, run `name_objects_by_layer.py` inside Rhino (`Tools > PythonScript > Run`). It will name all objects and export the two OBJ files automatically into the assets folder.

If you prefer to export manually:

| Step | Action |
|------|--------|
| 1 | Select all room volumes → Properties → set **Name** to room type (`bedroom`, `livingroom`, `kitchen`, `dining`, `corridor`, `stairs`, `storeroom`, `bathroom`, `balcony`) |
| 2 | Export selected → `type_k_rooms.obj` (enable **Export object names** in the dialog) |
| 3 | Select all aperture surfaces → set **Name** to door type (`door`, `passage`, `entrance_door`) |
| 4 | Export selected → `type_k_apertures.obj` |

This notebook builds the CellComplex and Graph. Continue in **03.1** to encode features and export to CSV.

## 1. Import libraries

In [ ]:
from topologicpy.Topology    import Topology
from topologicpy.Cell        import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Face        import Face
from topologicpy.Dictionary  import Dictionary
from topologicpy.Graph       import Graph
from topologicpy.Helper      import Helper

print('TopologicPy:', Helper.Version())

## 2. Set renderer

In [ ]:
renderer = "vscode"

## 3. Configuration

In [ ]:
import re
from pathlib import Path

# ── Change this to 'f' to process Type F ──────────────────────────
FLOOR_TYPE = 'f'
# ──────────────────────────────────────────────────────────────────

HERE    = Path.cwd()
ASSETS  = HERE.parent / '03_node_classification' / 'assets'
RESULTS = HERE.parent / '03_node_classification' / 'results'

ROOMS_OBJ     = ASSETS / f'type_{FLOOR_TYPE}_rooms.obj'
APERTURES_OBJ = ASSETS / f'type_{FLOOR_TYPE}_apertures.obj'

TOLERANCE = 0.01

def base_type(name):
    """Strip trailing _N suffix (e.g. bedroom_3 -> bedroom)."""
    return re.sub(r'_\d+$', '', name) if name else name

print(f'Floor type:    {FLOOR_TYPE.upper()}')
print('Rooms OBJ:    ', ROOMS_OBJ)
print('Apertures OBJ:', APERTURES_OBJ)

## 4. Build Cells from `type_k_rooms.obj`

In [ ]:
objects = Topology.ByOBJPath(str(ROOMS_OBJ))
print(f'{len(objects)} objects loaded from rooms OBJ\n')

cells     = []
selectors = []
skipped   = []

for obj in objects:
    d         = Topology.Dictionary(obj)
    full_name = Dictionary.ValueAtKey(d, 'name') or ''
    room_type = base_type(full_name)   # bedroom_3 → bedroom

    faces = Topology.Faces(obj)
    if not faces or len(faces) < 2:
        skipped.append((full_name, 'not enough faces'))
        continue

    cell = Cell.ByFaces(faces, tolerance=TOLERANCE)
    if cell is None:
        skipped.append((full_name, 'Cell.ByFaces failed — check volume is closed'))
        continue

    selector = Topology.InternalVertex(cell, tolerance=TOLERANCE)
    d_new    = Dictionary.ByKeyValue('room_type', room_type)
    selector = Topology.SetDictionary(selector, d_new)

    cells.append(cell)
    selectors.append(selector)
    print(f'  ✓  {full_name}  →  room_type="{room_type}"')

print(f'\nCells built: {len(cells)}')
if skipped:
    print('Skipped:')
    for s in skipped:
        print(f'  ✗  {s[0]}: {s[1]}')

## 5. Build aperture Faces from `type_k_apertures.obj`

In [ ]:
ap_objects = Topology.ByOBJPath(str(APERTURES_OBJ))
print(f'{len(ap_objects)} objects loaded from apertures OBJ\n')

apertures = []
skipped_a = []

for obj in ap_objects:
    d         = Topology.Dictionary(obj)
    full_name = Dictionary.ValueAtKey(d, 'name') or ''
    door_type = base_type(full_name)   # door_2 → door

    faces = Topology.Faces(obj)
    if not faces:
        skipped_a.append((full_name, 'no faces'))
        continue

    face  = faces[0]
    face  = Topology.RemoveCollinearEdges(face)
    d_new = Dictionary.ByKeyValue('door_type', door_type)
    face  = Topology.SetDictionary(face, d_new)
    apertures.append(face)
    print(f'  ✓  {full_name}  →  door_type="{door_type}"')

print(f'\nApertures built: {len(apertures)}')
if skipped_a:
    print('Skipped:')
    for s in skipped_a:
        print(f'  ✗  {s[0]}: {s[1]}')

## 6. Build CellComplex and transfer dictionaries

In [ ]:
cc = CellComplex.ByCells(cells, tolerance=TOLERANCE)
print('CellComplex type:', Topology.TypeAsString(cc))
print('Cells in complex:', len(Topology.Cells(cc)))

# Transfer room_type dicts from selectors to the CellComplex cells
cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranCells=True)
print('Dictionaries transferred.')

## 7. Add apertures

In [ ]:
cc = Topology.AddApertures(cc, apertures, exclusive=False, subTopologyType='Face', tolerance=TOLERANCE)
print('Apertures added.')

## 8. Create the Graph

In [ ]:
graph    = Graph.ByTopology(cc, direct=False, directApertures=True)
vertices = Graph.Vertices(graph)
edges    = Graph.Edges(graph)
print(f'Graph — vertices: {len(vertices)}  edges: {len(edges)}')

print('\n--- Vertex dictionaries (first 10) ---')
for v in vertices[:10]:
    d = Topology.Dictionary(v)
    print(f'  keys={Dictionary.Keys(d)}  values={Dictionary.Values(d)}')

print('\n--- Edge dictionaries (first 10) ---')
for e in edges[:10]:
    d = Topology.Dictionary(e)
    print(f'  keys={Dictionary.Keys(d)}  values={Dictionary.Values(d)}')

## 9. Visualise

In [ ]:
Topology.Show(cc, graph, renderer=renderer)

In [ ]:
Topology.Show(cc, *apertures, graph, renderer=renderer)